# 01 — Signatures & Predict: declare *what*, not *how*

**What you'll learn**

- Signatures: typed input/output contracts for LM calls — the "layer shape" of DSPy.
- `dspy.Predict`, the atomic module, and the `Prediction` objects it returns.
- Typed outputs (`bool`, `int`, `list[str]`, `Literal[...]`) without writing a parser.
- `dspy.inspect_history()` — how to always see the actual prompt.

*Time: ~20 min. Cost: < $0.02. Cached reruns are free.*

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import dspy
from dotenv import load_dotenv              # pip install -r requirements.txt if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead
# STRONG_MODEL = "openrouter/moonshotai/kimi-k2.6"
# STRONG_MODEL = "openrouter/openai/gpt-4o"

lm = dspy.LM(MODEL, max_tokens=1000)        # cache=True by default: reruns are ~free
dspy.configure(lm=lm)

### Phoenix observability (optional)

Run the next cell to stream this notebook's LM calls into a local [Phoenix](https://phoenix.arize.com) trace UI — or skip it; nothing below depends on tracing. It finds or starts the Phoenix server, prints the link, and names the session by date (`dspy-lab-YYYY-MM-DD`) so different days never mix. Notebook 11 tells the full story.

In [ ]:
# optional: Phoenix tracing (see notebook 11)
import obs

obs.enable_phoenix()

### MLflow observability (optional)

Run the next cell to record this notebook's LM calls with MLflow instead (or as well): logs land in a repo-local store that is safe against remnants and reruns, and the second line finds or starts the MLflow UI — click the printed link to browse them. Equally optional; equally explained in notebook 11.

In [ ]:
# optional: MLflow tracing (see notebook 11)
import obs

obs.enable_mlflow()
obs.mlflow_ui()            # prints the link where the traces appear

## The problem signatures solve

A prompt mixes together three different concerns: *what* the task is, *how* to coax the model into doing it, and *how to parse* what comes back. When you hand-write prompts, you maintain all three at once, forever, per model.

A **signature** isolates the first concern and makes it declarative. You state the fields going in, the fields coming out, and their types. Here is the anatomy:

```
     "context: list[str],  question: str   ->   answer: str"
      ------------------   -------------        -----------
        input field         input field         output field
        (name: type)        (name: type)        (name: type)
                                          --
                             "given these, produce those"
```

Names carry meaning (the model sees them), types constrain parsing, and — crucially — everything about *how* to prompt is left unspecified. That gap is filled mechanically today (notebook 02) and *optimized* later (notebooks 05–07). Keeping the what/how separation clean is what makes the optimization possible at all.

## The smallest possible program: one `Predict`

`dspy.Predict(signature)` turns a signature into a callable module. Inputs are passed as keyword arguments matching the input field names — always keywords, never positional.

In [ ]:
clf = dspy.Predict("sentence -> sentiment: bool")
clf(sentence="This chowder is a crime against the sea.")

> **What you should see:** `Prediction(sentiment=False)` — and note that `False` is a real Python `bool`, not the string `"False"`. You declared the type; DSPy instructed the model accordingly and parsed the reply. You wrote zero prompt text and zero parsing code.

Multiple typed outputs work the same way. This is structured extraction with no JSON wrangling:

In [ ]:
extract = dspy.Predict("bio -> name: str, age: int, hobbies: list[str]")

pred = extract(bio="Odo, 71, retired lightkeeper, spends his days birdwatching and grinding lenses.")
print(pred.name, "|", pred.age, "|", pred.hobbies)

> **What you should see:** something like `Odo | 71 | ['birdwatching', 'grinding lenses']` — with `pred.age` an `int` and `pred.hobbies` a real `list`.

The return value is a `dspy.Prediction`: output fields are attributes (`pred.name`), and the object prints nicely. Hold on to this class — in notebook 02 you'll learn a sneaky fact about it that the optimizers exploit.

## Trust, but verify: `inspect_history`

You did not write a prompt, but a prompt was certainly sent. DSPy never hides it:

In [ ]:
dspy.inspect_history(n=1)

> **What you should see:** the full last request — a system message that describes your fields and how to format the response (look for `[[ ## name ## ]]`-style field markers), your inputs as a user message, and the model's raw reply before parsing.
>
> Make `inspect_history` a reflex. Every abstraction in this course is transparent under it: whenever you wonder "but what did the model actually see?", the answer is one call away. Where those field markers come from is exactly the subject of notebook 02.

## Class-based signatures: when fields need explaining

Inline strings are perfect for quick calls. When the task needs instructions or field descriptions, write the signature as a class: the **docstring becomes the task instructions**, and `desc=` annotates individual fields.

In [ ]:
from typing import Literal

class Emotion(dspy.Signature):
    """Classify the dominant emotion in the text."""

    text: str = dspy.InputField()
    emotion: Literal["joy", "anger", "fear", "surprise"] = dspy.OutputField(
        desc="exactly one label, the single strongest emotion"
    )

dspy.Predict(Emotion)(text="The lens room door was open. It is never open.")

> **What you should see:** `Prediction(emotion='fear')` — the `Literal` type means the output is constrained to your four labels, so downstream code can rely on it.

A signature is an ordinary Python object you can inspect and transform. In particular, its *instructions are data*, not code:

In [ ]:
print(Emotion.instructions)
print(Emotion.input_fields)
print(Emotion.output_fields)

Stricter = Emotion.with_instructions(
    "Classify the dominant emotion. If two emotions compete, prefer the one driving the final sentence."
)
print(Stricter.instructions)

> **What you should see:** the docstring as `instructions`, dicts describing each field, and a *new* signature class carrying the replacement instructions.
>
> This is the most important sentence in this notebook: **the instructions are a piece of mutable data hanging off the signature.** In notebook 02 you'll see they are one of exactly two things every predictor carries as learnable state — and in notebooks 06–07, optimizers rewrite them for you, mechanically, better than you would by hand.

## Two syntaxes, one object

A fair question at this point: are inline strings and signature classes two different mechanisms? **No — they are two spellings of the same abstraction.** The string form is compiled into exactly the kind of class you just wrote by hand, with default instructions auto-generated from the field names. Nothing about the arrow syntax ever reaches the model (notebook 02 shows what does).

In [ ]:
sig = dspy.Signature("question -> answer")       # the string form builds a class...
print(type(sig) is type(Emotion), issubclass(sig, dspy.Signature))

print(sig.instructions)                          # ...with default instructions filled in

# The bridge between the spellings: a string signature plus instructions
# IS a class signature — this is equivalent to writing the class by hand.
OneWord = dspy.Signature("question -> answer").with_instructions(
    "Answer in exactly one word."
)
print(OneWord.instructions)

> **What you should see:** `True True` (same metaclass, same base — one kind of object), then the auto-written default instructions — `Given the fields 'question', produce the fields 'answer'.` — then your replacement.
>
> So the string is shorthand for a class, and a class is a string plus instructions and field notes. Choose by ergonomics — string for quick wiring, class when the task deserves stating up front — never by capability, because there is no capability difference. Whichever spelling you start from, `instructions` is the same slot, and the optimizers in notebooks 06–07 rewrite it either way.

Two practical notes before moving on:

- **Parsing can fail.** If the model replies in a shape that doesn't fit the declared types, DSPy raises `AdapterParseError` (rare with capable models, likelier with tiny local ones). Notebook 02 shows `JSONAdapter`, which makes structured outputs sturdier.
- **Old tutorials:** `dspy.InputField(prefix=..., format=...)` arguments are deprecated no-ops in 3.x — only `desc=` matters now. And there is no separate `TypedPredictor` anymore; plain `Predict` handles all the typing you saw here.

## Recap

| Concept | One-liner |
|---|---|
| signature | typed I/O contract: `"question -> answer"`, `"sentence -> sentiment: bool"` |
| class signature | docstring = instructions; `InputField`/`OutputField(desc=...)` for field notes |
| string vs class | one object, two spellings — the string compiles to a class with default instructions |
| `dspy.Predict(sig)` | the atomic module; call with keyword args matching input fields |
| `Prediction` | attribute access to typed outputs |
| `Signature.with_instructions` | instructions are data — the thing optimizers will learn |
| `dspy.inspect_history(n)` | always see the real prompt and raw reply |

## Exercises

1. Write `dspy.Predict("email -> is_spam: bool, reason: str")` and test it on one obvious spam and one borderline newsletter. Does `reason` justify the `bool`?
2. Convert it to a class signature whose instructions say to err on the side of *not* spam. Run both versions on the borderline email, then call `dspy.inspect_history(n=2)` and find exactly where the prompts differ.
3. Run `dspy.Predict("question -> answer: int")(question="What color is the sky?")`. What happens, and why is that the *right* behavior for a typed system?

**Next up:** where the prompt actually comes from — adapters — and the two learnable parameters hiding inside every predictor.